In [27]:
import warnings
warnings.filterwarnings("ignore")
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from fastapi.middleware.cors import CORSMiddleware


### Data Ingestion

In [28]:
def data_ingestion(sample_txt_file):

    # Text Loader
    from langchain_community.document_loaders import TextLoader
    text_loader = TextLoader(sample_txt_file, encoding="utf-8")
    text_loader
    # Load the document
    document = text_loader.load() # Gives the document structure with page_content and updated metadata
    print(type(document))
    return document
    



# Creating a txt file with the content of the document
os.makedirs("../data/test_files", exist_ok=True)
sample_txt_file = "../data/bridge_health_report.txt"
doc = data_ingestion(sample_txt_file)

<class 'list'>


### Chunking


In [29]:
def chunk(document , chunk_size = 100 , chunk_overlap = 20):
    # Split for better RAG performance
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size , chunk_overlap=chunk_overlap , length_function=len)
    split_doc = text_splitter.split_documents(document) # split_documents takes a list of documents and splits each document into smaller chunks based on the specified chunk_size and chunk_overlap. The resulting split_doc is a list of smaller Document objects, each containing a portion of the original document's content and metadata.
    print(f"Number of chunks created: {len(split_doc)}")  
    if split_doc:
        print("Example chunk:")
        print(f"Chunk content: {split_doc[0].page_content}") # page_content contains the actual text content of the chunk, which is a portion of the original document's content. This is the part that will be used for processing and retrieval in RAG.
        print(f"Chunk metadata: {split_doc[0].metadata}") # metadata contains the additional information about the chunk, such as its source, author, and creation date. This metadata is inherited from the original document and can be useful for organizing and retrieving chunks later on.
    return split_doc

chunks = chunk(doc)
chunks
        


Number of chunks created: 178
Example chunk:
Chunk content: BRIDGE HEALTH MONITORING REPORT
Generated   : 2026-04-10 15:49:27.409000
Total Windows : 50
Chunk metadata: {'source': '../data/bridge_health_report.txt'}


[Document(metadata={'source': '../data/bridge_health_report.txt'}, page_content='BRIDGE HEALTH MONITORING REPORT\nGenerated   : 2026-04-10 15:49:27.409000\nTotal Windows : 50'),
 Document(metadata={'source': '../data/bridge_health_report.txt'}, page_content='Total Windows : 50\nHealth State Distribution:\nhealth_state\nHealthy    30\nWatch      16'),
 Document(metadata={'source': '../data/bridge_health_report.txt'}, page_content='Watch      16\nWarning     4\n========================================'),
 Document(metadata={'source': '../data/bridge_health_report.txt'}, page_content='Bridge Sensor Report — Window 1\nTimestamp          : 2026-04-10 15:43:46.796000'),
 Document(metadata={'source': '../data/bridge_health_report.txt'}, page_content='Health State       : Watch\nFiltered Strain    : 47.9474 mV\nFiltered Distance  : 5.1177 cm'),
 Document(metadata={'source': '../data/bridge_health_report.txt'}, page_content='Total RMS Accel    : 0.940182 g\nAcceleration Anomaly : False\nStrain 

### Embedding 

In [30]:
class EmbeddingManager:
    #Handles document embedding generation using SentenceTransformer
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):  #HuggingFace model name for sentence embeddings

        self.model_name = model_name
        self.model = None
        self._load_model() # Load the model during initialization... uses _ to indicate it's a private method not meant to be accessed outside the class.
    
    def _load_model(self):
        try:
            print(f"Loading model '{self.model_name}'...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model '{self.model_name}' loaded successfully..Model dimensionality: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model '{self.model_name}': {e}")

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:   # Generates embeddings for a list of texts and returns them as a numpy array. The method checks if the model is loaded before attempting to generate embeddings. If the model is not loaded, it raises a ValueError. If an error occurs during embedding generation, it catches the exception and prints an error message, returning an empty numpy array.
        if not self.model:
            raise ValueError("Model not loaded. Cannot generate embeddings.") 
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print("Embeddings generated successfully.")
        return embeddings


        
embedding_manager = EmbeddingManager()
embedding_manager

Loading model 'all-MiniLM-L6-v2'...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9032.27it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model 'all-MiniLM-L6-v2' loaded successfully..Model dimensionality: 384


### VectorDB

In [31]:
class VectorStore:
    #Manages document embeddings in a ChromaDB vector store
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        #Initialize ChromaDB client and collection
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = [] # Unique IDs for each document, generated using uuid to ensure uniqueness across different runs and avoid collisions. The ID format includes a prefix "doc_", a unique hexadecimal string from uuid, and the document index for traceability.
        metadatas = []  # Metadata for each document, extracted from the original document's metadata and augmented with additional information such as the document index and content length. This metadata can be useful for organizing and retrieving documents later on.
        documents_text = []  # The actual text content of each document, extracted from the page_content attribute of the LangChain Document objects. This is the part that will be stored in the vector store and used for retrieval in RAG.
        embeddings_list = []  # The embeddings for each document, converted to a list format suitable for storage in ChromaDB. The embeddings are generated using the SentenceTransformer model and are stored as lists of floats in the vector store.
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique IDs
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore


#Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

# Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

# Store in the vector database
vectorstore.add_documents(chunks,embeddings)

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 548
Generating embeddings for 178 texts...


Batches: 100%|██████████| 6/6 [00:00<00:00, 15.54it/s]

Embeddings generated successfully.
Adding 178 documents to vector store...
Successfully added 178 documents to vector store
Total documents in collection: 726


### RAG Retrieval

In [32]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings(query)[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


rag_retriever=RAGRetriever(vectorstore,embedding_manager)
rag_retriever

rag_retriever.retrieve("what is Python?", top_k=3, score_threshold=0.5)

Retrieving documents for query: 'what is Python?'
Top K: 3, Score threshold: 0.5
Generating embeddings for 15 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 95.34it/s]

Embeddings generated successfully.
Error during retrieval: Collection expecting embedding with dimension of 384, got 1


[]

### Integrating VectorDB Context Pipeline with LLM output

In [33]:
# Initialize the Groq LLM (set your GROQ_API_KEY in environment)
load_dotenv(override=True) # Load environment variables from .env file, with override=True to ensure that any existing environment variables with the same names are overwritten by the values from the .env file. This is useful for development and testing purposes, allowing you to easily manage and switch between different sets of environment variables without affecting the global environment.
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024)

# 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=5):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else "" # If results are found, it extracts the 'content' field from each retrieved document and joins them into a single string with two newlines ("\n\n") as separators. If no results are found, it sets the context to an empty string.
    if not context:
        return "No relevant context found to answer the question." 
    
    # Generate the answwer using GROQ LLM
    
    prompt = """You are BridgeAI, an AI assistant exclusively dedicated to bridge structural health monitoring and engineering.

    Your knowledge scope is strictly limited to:
    - This bridge's live sensor data and health reports
    - General politeness and professionalism in communication
    - Bridge engineering, design, and structural analysis
    - Vibration analysis, fatigue, and predictive maintenance
    - Civil and structural engineering principles related to bridges

    You are NOT permitted to answer questions outside this scope under any circumstances. Do not attempt to be helpful on out-of-scope topics. Do not provide partial answers. Do not acknowledge the out-of-scope topic at all.

    Here is the current live data and report context for this specific bridge:
    {context}

    Question: {query}

    Instructions:
    1. If the question is about the current status, health scores, or recent reports, answer strictly using the provided context.
    2. If the question is a general bridge or structural engineering question, use your structural engineering knowledge to answer it.
    3. If combining both, clearly state what is from the actual bridge data versus what is a general engineering principle.
    4. If the question is outside your defined scope — meaning it is not about bridges, structural engineering, or this bridge's health data — respond with exactly this and nothing else: "I can only assist with bridge health monitoring and structural engineering questions. This question is outside my scope."

    Answer:"""
        
    response=llm.invoke([prompt.format(context=context,query=query)]) # The invoke method of the ChatGroq class is called with a list containing the formatted prompt. The prompt includes the retrieved context and the user's query, instructing the model to generate a concise answer based on the provided information. The response from the model is expected to contain the generated answer, which is then returned as the output of the rag_simple function.
    return response.content




### Frontend Connection

In [34]:
app = FastAPI() # this initializes a FastAPI application, which will be used to create API endpoints for handling incoming requests. The app variable is an instance of the FastAPI class, and it serves as the main entry point for defining routes and handling HTTP requests in the application.

app.add_middleware(   # This block adds CORS (Cross-Origin Resource Sharing) middleware to the FastAPI application, allowing it to handle requests from different origins. The configuration allows all origins, credentials, methods, and headers, which is useful during local development when the frontend and backend may be running on different ports. In a production environment, you would typically restrict these settings for better security.
    CORSMiddleware,
    allow_origins=["*"],  # Allows all frontend ports during local development
    allow_credentials=True,
    allow_methods=["*"],  # Allows POST, GET, etc.
    allow_headers=["*"],  # Allows all headers
)

my_queries = ["what is sensor data now?"]  # This list will store user queries received through the API endpoint. Each time a new query is submitted, it will be appended to this list, allowing us to keep track of all queries and process them in a loop later on.

class QueryRequest(BaseModel):
    user_query: str

@app.post("/submit-query/")
async def submit_query(request: QueryRequest):
    my_queries.append(request.user_query)
    return {"message": "Success", "queries": my_queries}


if(len(my_queries)==2):
    my_queries.remove(my_queries[0])  # This condition checks if the my_queries list has 2 queries (the initial one and the newly added one). If it does, it removes the first query (the initial one) to ensure that only the latest query is kept in the list. This is a simple way to manage the list of queries and prevent it from growing indefinitely during testing. In a real application, you would likely want a more robust solution for managing queries, such as a database or a queue.


# Run the RAG pipeline
answer = rag_simple(my_queries, rag_retriever, llm, top_k=3)
print(f"Answer: {answer}\n{'-'*40}\n")

Retrieving documents for query: '['what is sensor data now?']'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 106.86it/s]

Embeddings generated successfully.
Retrieved 2 documents (after filtering)


Answer: Based on the provided context, I can see that the bridge is equipped with advanced sensors and machine learning algorithms. 

According to the current live data, the sensor readings are as follows:

- Temperature: 22.5°C
- Vibration levels: 0.05g (within normal limits)
- Strain levels: 0.002 (within normal limits)
- Acceleration: 0.1m/s² (within normal limits)

The health report indicates that the bridge's structural integrity is within acceptable limits, with no signs of distress or damage. The machine learning algorithms have not detected any anomalies or potential issues that require immediate attention.

Please note that these values are subject to change and may not reflect the current status at the time of your inquiry. For the most up-to-date information, I recommend checking the live data feed.
----------------------------------------

